# Three dimensions

Hierarchies, regridding, the spaces and the solvers work in 3D as in 2D. A hanging node can now
sit on a coarse edge or in the middle of a coarse face, and so can a hanging edge; faces are a
space of their own (lowest-order Raviart–Thomas), linked to the edges by the curl. This
notebook goes through the chain nodes → edges → faces → cells on a refined box and solves
Poisson's equation and the curl–curl equation on it.

In [ ]:
import numpy as np

from patchwork import (
    AMRHierarchy,
    CompositeEdges,
    CompositeFaces,
    CompositeNodes,
    CurlCurl,
    Dirichlet,
    NodalPoisson,
    function_tags,
)

## A refined box

`regrid` covers the tagged cells with boxes (Berger–Rigoutsos, in 3D), two levels around a ball.

In [ ]:
mesh = AMRHierarchy(base_shape=(8, 8, 8))


def ball(x, y, z):
    return (x - 0.5) ** 2 + (y - 0.45) ** 2 + (z - 0.55) ** 2 < 0.2**2


mesh = mesh.regrid(function_tags(mesh, ball, levels=2), max_patch_size=8)
for level in range(mesh.max_level + 1):
    print(f"level {level}: {len(mesh.patches_at_level(level))} patches")

## Nodes and the nodal Poisson

A node on a coarse face takes the bilinear interpolant of the face's corners. With the
consistent (Q1) mass, a trilinear potential is solved exactly, across the patch faces too; the
lumped scheme (the seven-point stencil) converges at second order.

In [ ]:
nodes = CompositeNodes(mesh)
print(f"{nodes.num_nodes} nodes, {nodes.hanging.sum()} hanging")


def trilinear(x, y, z):
    return 1.0 + x - 2.0 * y + 3.0 * z + 5.0 * x * y * z


sides = ("x_lower", "x_upper", "y_lower", "y_upper", "z_lower", "z_upper")
faces = {side: Dirichlet(trilinear) for side in sides}
for mass in ("lumped", "consistent"):
    solver = NodalPoisson(mesh, faces=faces, nodes=nodes, mass=mass)
    x, y, z = nodes.coordinates.T
    error = np.abs(solver.solve(np.zeros_like(x)) - trilinear(x, y, z)).max()
    print(f"{mass:>10}: largest error {error:.1e}")

On large 3D meshes `backend="petsc"` solves the same operator with conjugate gradients and
BoomerAMG, with the rows distributed over the ranks.

## Edges, faces and the exact sequence

The gradient maps nodes to edges, the curl edges to faces and the divergence faces to cells. On
the lattice these are incidence matrices, so `C G = 0` and `D C = 0` hold exactly, and the
constraints of the hanging edges and faces commute with them.

In [ ]:
edges = CompositeEdges(nodes)
cells_faces = CompositeFaces(edges)
G, C, D = edges.gradient, cells_faces.curl, cells_faces.divergence
T, T1, T2 = nodes.reduction, edges.reduction, cells_faces.reduction
print(f"{edges.num_edges} edges, {cells_faces.num_faces} faces")
print("|C G| =", abs(C @ G).max(), " |D C| =", abs(D @ C).max())
print("|C T1 - T2 C T1| =", abs(C @ T1 - T2 @ C @ T1).max())

## The curl–curl equation

`CurlCurl` solves `curl(nu curl A) + beta A = J` on the edges, here with `A = (sin πy sin πz,
0, 0)`, whose tangential component vanishes on the box (electric faces). `solver="auto"` takes
hypre's AMS when petsc4py is built with hypre, and the LU otherwise.

In [ ]:
solver = CurlCurl(mesh, beta=1.0, edges=edges, solver="auto")
middle = 0.5 * (
    nodes.coordinates[edges.ends[:, 0]] + nodes.coordinates[edges.ends[:, 1]]
)
shape = np.sin(np.pi * middle[:, 1]) * np.sin(np.pi * middle[:, 2])
exact = np.where(edges.direction == 0, shape * edges.length, 0.0)
potential = solver.solve(solver.load((2.0 * np.pi**2 + 1.0) * exact))
difference = T1 @ (potential - exact)
print(f"solver {solver.solver}, {solver.free.size} unknowns")
print(
    f"relative error {np.sqrt(difference @ edges.mass() @ difference / (exact @ edges.mass() @ exact)):.1e}"
)